# EDA


### 0. Check files and read in data
- Check that the files are uploaded
- Read in the CSV

In [0]:
# Check files

dbutils.fs.ls("/Volumes/marathos/default/raw")

In [0]:
# Read in data 

DATA_PATH = "/Volumes/marathos/default/raw"

df_raw = spark.read.csv(f"{DATA_PATH}/TWO_CENTURIES_OF_UM_RACES.csv", header=True, inferSchema=True)

## 1. Initial EDA

#### Columns & rows

- Number of rows
- Number of columns
- Schema

In [0]:

df_raw.count()
# Find out number of rows

len(df_raw.columns)
# Find out number of columns

print(f"Number of rows:{df_raw.count()}")
print(f"Number of columns:{len(df_raw.columns)}")

In [0]:
df_raw.columns
# See all column names

In [0]:
df_raw.display(100)

#### Schema

In [0]:
df_raw.printSchema()
# See all column names and types

In [0]:
# Check what Event dates look like

df_raw.select("Event dates").distinct().limit(20).display()

#### Descriptive summary

In [0]:
# Descriptive summary of numerical fields

df_raw.select(
    "Athlete ID",
    "Athlete year of birth",
    "Event number of finishers",
    "Year of event"
).describe().display()




In [0]:
df_raw.select("Athlete average speed", "Athlete performance", "Event distance/length").display()

### Explore the data

#### Events

In [0]:
from pyspark.sql.functions import countDistinct

df_raw.select("Event name").distinct().display()
# View list of all unique events 

df_raw.groupBy("Event name") \
    .agg(countDistinct("Year of event").alias("number_of_years")) \
    .orderBy("number_of_years", ascending=False) \
    .display()
# View reaccuring events

unique_events = df_raw.select("Event name").distinct().count()
print(f"Number of unique events:{unique_events}")
# Count number of unique events



#### Nulls

In [0]:
# Counts number of null values in each column
from pyspark.sql.functions import col, sum as spark_sum

null_count = df_raw.select(
    [spark_sum(col(columns).isNull().cast("int")).alias(columns) for columns in df_raw.columns
])

# Display columns with null values
null_count = null_count.collect()[0].asDict()
[(column, nulls) for column, nulls in null_count.items() if nulls > 0]

In [0]:

# Display rows with nulls for ´Athlete age category´
df_raw_nulls = df_raw.filter(col("Athlete age category").isNull())

df_raw_nulls.display()

In [0]:
# Find connectins between null values

# 1. Nulls per year
df_raw_nulls.groupBy("Year of event").count().orderBy("Year of event").display()

# 2. Nulls per event
df_raw_nulls.groupBy("Event name").count().orderBy("count", ascending=False).display()


In [0]:
# 3. Check if Athlete year of birth and Athlete age category have null on same rows

only_birth_null = df_raw.filter(
    col("Athlete year of birth").isNull() & 
    col("Athlete age category").isNotNull()
).count()

only_age_null = df_raw.filter(
    col("Athlete year of birth").isNotNull() & 
    col("Athlete age category").isNull()
).count()

both_null = df_raw.filter(
    col("Athlete year of birth").isNull() & 
    col("Athlete age category").isNull()
).count()

print(f"Only birth null: {only_birth_null}")
print(f"Only age null: {only_age_null}")
print(f"Both null: {both_null}")


In [0]:
# Look at athlete club 

df_raw_nulls_club = df_raw.filter(col("Athlete club").isNull())

df_raw_nulls_club.display()

# Find connectins between null values

# 1. Nulls per year
df_raw_nulls_club.groupBy("Year of event").count().orderBy("Year of event").display()


# 2. Nulls per event
df_raw_nulls_club.groupBy("Event name").count().orderBy("count", ascending=False).display()

# 3. Nulls per country
df_raw_nulls_club.groupBy("Athlete country").count().orderBy("count", ascending=False).display()

In [0]:
# How many % nulls for Athlete club in Two Oceans Marathon

total_two_oceans = df_raw.filter(col("Event name") == "Two Oceans Marathon (RSA)").count()
null_two_oceans = df_raw_nulls_club.filter(col("Event name") == "Two Oceans Marathon (RSA)").count()

print(f"Totalt: {total_two_oceans}")
print(f"Nulls: {null_two_oceans}")
print(f"Andel: {round(null_two_oceans / total_two_oceans * 100, 1)}%")



In [0]:
from pyspark.sql.functions import count, sum as spark_sum, round

df_raw.filter(col("Event name") == "Two Oceans Marathon (RSA)") \
    .groupBy("Year of event") \
    .agg(
        count("*").alias("total"),
        spark_sum(col("Athlete club").isNull().cast("int")).alias("nulls")
    ) \
    .withColumn("null_pct", round(col("nulls") / col("total") * 100, 1)) \
    .orderBy("Year of event") \
    .display()

In [0]:
# Look at remaning colums with null values

null_columns = [ 
    "Athlete country", 
    "Athlete gender", 
    "Athlete performance"
]

for columns in null_columns:
    df_raw.filter(col(columns).isNull()) \
        .groupBy("Event name") \
        .count() \
        .orderBy("count", ascending=False) \
        .display()

#### Dates

In [0]:
df_raw.select("Event dates").distinct().limit(20).display()

#### Age distribution

In [0]:
df_raw.groupBy("Athlete year of birth") \
    .count() \
    .orderBy("count", ascending=False) \
    .display()

#### Countries

In [0]:
df_raw.groupBy("Athlete country") \
    .count() \
    .orderBy("count", ascending=False) \
    .display()

### Data profiling

In [0]:
df_raw.display(10)

#### Categorical columns

In [0]:
# Check unique values and amount of rows

categorical_cols = [
    "Athlete gender",
    "Athlete age category", 
    "Event distance/length"
]

for c in categorical_cols:
    print(f"\n{'='*40}")
    print(f"  {c}")
    print(f"{'='*40}")
    
    print(f"\nUnique values: {df_raw.select(c).distinct().count()}")
    
    df_raw.groupBy(c) \
        .count() \
        .orderBy("count", ascending=False) \
        .show(40)

In [0]:
df_raw.filter(col("Athlete age category") == "F35").display()

In [0]:
df_raw.filter(col("Athlete gender") == "X").display()

In [0]:
# Check to find all the unique values for distance/length - extract the unit

from pyspark.sql.functions import regexp_extract

df_raw.select(
    regexp_extract(col("Event distance/length"), "[a-zA-Z]+", 0).alias("unit")
) \
    .groupBy("unit") \
    .count() \
    .orderBy("count", ascending=False) \
    .show()

In [0]:
# Investigate the unit types that don't fit the standard

for unit in ["None", "m", "x", ""]:
    print(f"\n--- unit: '{unit}' ---")
    df_raw.filter(
        regexp_extract(col("Event distance/length"), "[a-zA-Z]+", 0) == unit
    ).select("Event distance/length", "Event name") \
        .distinct() \
        .show(10)

In [0]:
df_raw.filter(
    col("Event distance/length").isin(["6:40", "07:35", "71.5"])
).select("Event distance/length", "Event name", "Athlete performance") \
    .distinct() \
    .show()

In [0]:
# Countries

df_raw.select("Athlete country") \
    .distinct() \
    .orderBy("Athlete country") \
    .show(200)


#### Numerical outliers

In [0]:
df_raw.filter(
    (col("Athlete year of birth") < 1900) | 
    (col("Athlete year of birth") > 2010)
).select("Athlete year of birth").groupBy("Athlete year of birth") \
    .count() \
    .orderBy("Athlete year of birth") \
    .show(20)

df_raw.filter(
    (col("Year of event") < 1950) | 
    (col("Year of event") > 2023)
).groupBy("Year of event") \
    .count() \
    .orderBy("Year of event") \
    .show(20)

In [0]:
# Outliers in Athlete year of birth
df_raw.filter(
    (col("Year of event") - col("Athlete year of birth") < 10) |
    (col("Year of event") - col("Athlete year of birth") > 100)
).count()

In [0]:
# Event number of finishers = 0
df_raw.filter(col("Event number of finishers") == 0) \
    .groupBy("Event name") \
    .count() \
    .orderBy("count", ascending=False) \
    .show(20)

In [0]:
df_raw.filter(col("Event number of finishers") == 0).display()

df_raw.filter(col("Event name") == "Ultra trail Dinarides - King's Race (CRO)").display()

#### Format columns

In [0]:
# Athlete performance - se unique formats
df_raw.select("Athlete performance") \
    .distinct() \
    .orderBy("Athlete performance") \
    .show(30)

df_raw.select("Athlete performance") \
    .distinct() \
    .orderBy("Athlete performance", ascending=False) \
    .show(30)

In [0]:
# Check nr of rows with "d" in Athlete performance

from pyspark.sql.functions import regexp_extract

df_raw.filter(
    col("Athlete performance").rlike("^[0-9]+d")
).count()

In [0]:
# Check if it correlates with the rows that have "d" as unit
df_raw.filter(
    col("Athlete performance").rlike("^[0-9]+d") &
    ~col("Event distance/length").rlike("d$")
).count()

In [0]:
# CLoser look at the events 
df_raw.filter(
    col("Athlete performance").rlike("^[0-9]+d") &
    ~col("Event distance/length").rlike("d$")
).groupBy("Event distance/length", "Event name") \
    .count() \
    .orderBy("count", ascending=False) \
    .show(20)

In [0]:
# Check null values 
df_raw.filter(
    (col("Athlete performance") == "0:00:00 h") |
    (col("Athlete performance") == "0.000 km") |
    (col("Athlete performance") == "0:00:00 km")
).count()

# Check null value deviation 
df_raw.filter(
    (col("Athlete performance") == "0:00:00 h") |
    (col("Athlete performance") == "0.000 km") |
    (col("Athlete performance") == "0:00:00 km")
).groupBy("Athlete performance") \
    .count() \
    .show()

In [0]:
# Event dates - check unique formats
df_raw.select("Event dates") \
    .distinct() \
    .orderBy("Event dates") \
    .show(30)

df_raw.select("Event dates") \
    .distinct() \
    .orderBy("Event dates", ascending=False) \
    .show(30)

In [0]:
# Check nr of rows with 00:00

df_raw.filter(col("Event dates").startswith("00.00")).count()

In [0]:
# Athlete average speed - see unique formats
df_raw.select("Athlete average speed") \
    .distinct() \
    .orderBy("Athlete average speed") \
    .show(30)

df_raw.select("Athlete average speed") \
    .distinct() \
    .orderBy("Athlete average speed", ascending=False) \
    .show(30)

In [0]:
# Check nr of rows with avg. speed = 0 or > 500
df_raw.filter(
    (col("Athlete average speed") == "0") |
    (col("Athlete average speed") == "0.0")
).count()

df_raw.filter(
    col("Athlete average speed").cast("double") > 500
).count()

In [0]:
df_raw.filter(
    col("Athlete average speed").rlike("[a-zA-Z:]")
).select("Athlete average speed", "Athlete performance", "Event distance/length") \
    .distinct() \
    .show(50)

In [0]:
df_raw.filter(
    col("Athlete average speed").rlike("^[0-9]+:[0-9]+:[0-9]+$")
).count()


In [0]:
df_raw.filter(
    col("Athlete average speed").rlike("^[0-9]+:[0-9]+:[0-9]+$")
).groupBy("Event distance/length") \
    .count() \
    .orderBy("count", ascending=False) \
    .show(10)

In [0]:
df_raw.filter(
    col("Athlete average speed").rlike("^[0-9]+:[0-9]+:[0-9]+$")
).select(
    "Athlete average speed",
    "Athlete performance",
    "Event distance/length"
).show(20)

In [0]:
# Check 9999+ values

from pyspark.sql.functions import expr

df_raw.filter(
    expr("try_cast(`Athlete average speed` AS double)") > 500
).groupBy("Athlete average speed") \
    .count() \
    .orderBy("count", ascending=False) \
    .show(20)

In [0]:
df_raw.filter(
    expr("try_cast(`Athlete average speed` AS double)") > 500
).groupBy("Event name", "Event distance/length") \
    .count() \
    .orderBy("count", ascending=False) \
    .show(10)

In [0]:
df_raw.filter(
    expr("try_cast(`Athlete average speed` AS double)") > 500
).count()

In [0]:
df_raw.display(20)

Frågor: 

#### Bonus

- country data - either use LLM to simulate good guess of abbreviations or find a real one. Ingest this data in and use it to enhance the downstream layers.- detta ska vara en egen kolumn alltså, i vilket steg ska det in? 
- use LLM to create a new marathon with data and place it in to unity catalog and stream it
schedule the pipeline - baar en rad? hur många deltagare? i vilket steg görs detta? 
- create a date dimension - here you can use the starting date, but note that the data is very tedious - vad menas med detta och i vilket steg görs det?


#### skriv om 

- är det något från bonus vi måste tänka på nu innan vi går vidare?
- behöver vi kolla efter ogiltiga tecken i ec. clubs? syriliska



# Findings summary

### Null Analysis

**Athlete Age Data**

- `Athlete year of birth`: 588,161 nulls | `Athlete age category`: 584,938 nulls
- 584,740 rows (~8%) are missing **both** columns simultaneously → systematic missing data, not random
- Only 3,421 rows missing `year of birth` alone, and 198 missing `age category` alone
- No clear pattern linked to a specific year or event
- Age category could potentially be calculated for the 3,421 rows where `year of birth` exists, but given they represent only ~0.05% of total data the impact on any average age calculation would be negligible
- Spark's `avg()` automatically excludes nulls → ~8% of athletes excluded from any age calculations

**Athlete Club**

- 2,826,373 nulls (~38% of total data)
- Two causes identified:
  - **Historical data (pre-2001):** 100% nulls — club data was not collected
  - **Modern data (2001+):** 5-23% nulls — athlete has no club affiliation, which is normal for recreational runners
- Two Oceans Marathon (RSA) accounts for 129,684 nulls — largest single contributor
- Nulls are **meaningful** and should not be dropped → recommend replacing with `"Unknown"`

**Athlete Country, Gender & Performance**
- `Athlete country`: 3 nulls | `Athlete gender`: 7 nulls | `Athlete performance`: 2 nulls
- Combined, these represent 12 rows out of 7,461,195 (~0.0002%) — no clear pattern or event connection


---

### Descriptive Summary - Anomalies Found

- `Athlete year of birth`: min `1193`, max `2021` → clearly invalid values, filter out unrealistic birth years in Silver
- `Event number of finishers`: min `0` → likely missing data stored as 0, investigate in Silver
- `Year of event`: min `1798` → invalid, filter out in Silver
- `Athlete ID`: min `0` → verify if 0 is a valid ID or placeholder for missing data.

---

### Data Profiling

#### Categorical Columns

**Athlete gender**
- 4 unique values: `M` (6,035,358), `F` (1,425,784), `X` (46), `NULL` (7)
- `X` appears to be non-binary gender registration, introduced by some events from 2013 onwards — treat as valid value

**Athlete age category**
- 38 unique values, all following `M`/`W` + age group pattern (e.g. `M23`, `W40`)
- One exception: `F35` (1 row, female born 1986, event 2022) — incorrect format, map to `W35`

**Event distance/length**
- 2,160 unique values with multiple unit inconsistencies identified.

#### Numerical anomalies

**Year of event & Event number of finishers**

- `Year of event`: dataset contains legitimate historical events dating back to 1798 — no lower bound filter applied. Rows where `Year of event` > 2023 are invalid and will be dropped
- `Event number of finishers`: "Ultra trail Dinarides - King's Race (CRO)" has only 10 rows, all with 0 finishers — likely a cancelled or incorrectly registered event.

**Athlete year of birth — age validation**
- 291 rows where athlete age at event is < 10 or > 100 years → clearly invalid

#### Format columns

**Athlete performance — multi-day format**
- 62,727 rows with format `Xd HH:MM:SS h` — legitimate data for multi-day ultramarathons 
  (e.g. Tor des Géants 330km, Moab 240 238mi)
- These are NOT correlated with `d`-unit events — they are distance events that simply take several days
- Silver transformation must handle this format: `Xd HH:MM:SS` → convert to total seconds

**Event distance/length — additional format anomalies**
- `250.7km/6Etappen`, `250km/6Etappen` — multi-stage race format → investigate if these should be kept or dropped

**Event dates**
- 505 rows with format `00.00.YYYY` — day and month unknown, only year available
- Cannot be cast to full date → extract year only in Silver, store as partial date

** Athlete average speed**
- 7,676 rows contain time format (`HH:MM:SS`) instead of speed value
- All belong to time-based events (`h`) — elapsed time has been stored in speed column
- Data is meaningful: cross-referencing with `Athlete performance` confirms the pattern
  e.g. `18:00:00` speed + `131.4 km` performance on a 24h event = athlete ran for 18 hours
- Correct average speed can be calculated in Silver: `performance_km / elapsed_hours`

- Values > 500 appear to be in m/h instead of km/h
- Affects major events including Two Oceans Marathon, Comrades Marathon, 100km Lauf Biel
- Conversion needed: `athlete_average_speed / 1000` for these rows in Silver
- Verify by cross-checking: 8,451 m/h = 8.451 km/h → reasonable for a 56km race

## Actions Required for Cleaning

### Null handling

- `Athlete club`: replace nulls with `"Unknown"` — do **not** drop rows
- `Athlete year of birth` / `Athlete age category`: leave nulls as-is, Spark handles them automatically in aggregations
- `Athlete country`, `Athlete gender`, `Athlete performance`: drop the 12 affected rows — negligible impact on dataset

### Type casting
- `Athlete year of birth`: double → integer
- `Athlete average speed`: string → double
- `Event dates`: extract start date from inconsistent format (`06.01.2018` / `24.-26.03.2018`) → standardize to `YYYY-MM-DD` and cast to `date`
- **Column names:** rename all columns to snake_case in **Silver** 
  for consistency with SQL and dimensional modeling in Task 4
- `Event dates`: 505 rows with `00.00.YYYY` format → extract year only, 
  remaining rows extract start date and cast to `date` in Silver

### Transformations

- `Athlete performance`: string in two formats depending on event type.
Split value and unit, then convert:
  - time-based (`8:13:16 h`) → convert to seconds (integer)
  - distance-based (`60.375 km`) → extract float value


- `Athlete average speed` (time-based events): rename to `athlete_elapsed_time` for `h`-events, 
  then calculate correct `athlete_average_speed` = `performance_km / elapsed_hours` in Silver
- `Athlete average speed`: convert m/h values to km/h (`/ 1000`) for rows where 
  value > 500 in Silver

### Data quality

#### Invalid rows to drop
- `Event distance/length` unit is `d` (days): 12,190 rows
- `Event distance/length` is `4x52km` (relay format): 4 rows
- `Event distance/length` is `None` (missing as string): 1,053 rows
- `Event distance/length` is `6:40` with performance `0:00:00 km`: invalid data
- `Year of event`: drop rows where year > 2023 (future events) — check count first
- `Event number of finishers`: drop 10 rows for "Ultra trail Dinarides - King's Race (CRO)" — all rows show 0 finishers, likely cancelled or incorrectly registered event

#### Standardization
- `Athlete age category`: map `F35` → `W35`
- `Event distance/length` units: standardize `Km`/`K`/`k` → `km` and `Miles`/`miles`/`Mile`/`mile` → `mi`
- `Event distance/length`: map `45.1m` → `45.1mi`, `71.5` → `71.5km`, `07:35` → `07:35h`
- `Athlete performance`: handle multi-day format `Xd HH:MM:SS h` → convert to total seconds in Silver
- `Event distance/length`: investigate multi-stage format `XXXkm/6Etappen` — decide keep or drop

#### Outliers

- `Athlete year of birth`: filter out unrealistic values (min: `1193`, max: `2021`) in **Silver**
- `Year of event`: filter out invalid years (min: `1798`) in **Silver**
- `Event number of finishers`: investigate rows where value is `0` — likely missing data stored as 0 in **Silver**
- `Athlete ID`: verify if `0` is a valid ID or placeholder for missing data in **Silver**

#### Validity checks
- If `event_distance_unit` = `km` or `mi` → `athlete_performance` must be in `h` format
  → drop rows where this does not match
- If `event_distance_unit` = `h` → `athlete_performance` must be in `km` format
  → drop rows where this does not match
  - Verify `Year of event` matches year extracted from `Event dates`


### New columns to create
- `event_type`: derive from `event_distance_unit` → `"distance"` (km/mi) or `"time"` (h) — required for gold views (Task 5) and validity checks
- `event_distance_value`: extract numeric value from `event_distance_length`
- `event_distance_unit`: extract unit from `event_distance_length`
- `event_id`: dense_rank() on `event_name`
- `athlete_id`: verify if existing column is sufficient or if new one needs to be created
- `result_id`: unique key per row

- **Dimensional modeling reminder (Task 4):** for time-based events (e.g. `6h`), the duration is constant for all athletes and belongs in `dim_event` — `Athlete performance` for these events represents distance covered, which belongs in `fct_results`. Keep this separation in mind when designing the dimensional model in Silver/Gold
